# TradeStation API Usage Examples

This notebook demonstrates how to use the TradeStation API library.

## Prerequisites
- Create a `.env` file with your credentials (see `.env.example`)
- Install dependencies: `pip install -r requirements.txt`

## Environment Selection
Set the environment below:
- `'sim'` - Paper trading (simulation) - **SAFE**
- `'prod'` - Live trading - **REAL MONEY** ⚠️


In [6]:
# ============================================================
# CONFIGURATION - Set environment here
# ============================================================

# Choose environment: 'sim' or 'prod'
ENVIRONMENT = 'sim'  # CHANGE THIS to 'prod' for live trading (REAL MONEY)

print(f"Environment: {ENVIRONMENT}")
print(f"⚠️  {'SIMULATION MODE - Paper trading' if ENVIRONMENT == 'sim' else 'PRODUCTION MODE - REAL MONEY'}")


Environment: sim
⚠️  SIMULATION MODE - Paper trading


In [7]:
# ============================================================
# SETUP - Import libraries and initialize API
# ============================================================

# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

# Add the parent directory to path so we can import the api module
sys.path.insert(0, str(Path().absolute().parent))

# Import the TradeStation API
from tradestation.api import TradeStationAPI

# Initialize the API with the selected environment
# This will load credentials from .env and set the appropriate API URL
api = TradeStationAPI(ENVIRONMENT)

# Get the account ID from configuration
account_id = api.config.account_id

print(f"\n✓ API initialized successfully")
print(f"✓ Base URL: {api.config.base_url}")
print(f"✓ Account ID: {account_id}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Loaded configuration from: .env
✓ Environment: sim

✓ API initialized successfully
✓ Base URL: https://sim-api.tradestation.com
✓ Account ID: SIM2977785M


---
## Market Data

Get historical and real-time market data.


In [ ]:
# ============================================================
# Get Bar Chart Data (Historical Prices)
# ============================================================

# Get last 10 bars of 5-minute data for ES (S&P 500)
symbol = 'ES'
bars = api.market_data.get_bars(
    symbol=symbol,
    interval=5,          # 5-minute intervals
    unit='Minute',       # Unit of time
    bars_back=10         # Number of bars to retrieve
)

print(f"Retrieved {len(bars.get('Bars', []))} bars for {symbol}")
print(f"\nFirst bar:")
print(json.dumps(bars.get('Bars', [])[0] if bars.get('Bars') else {}, indent=2))
print(f"\nLast bar:")
print(json.dumps(bars.get('Bars', [])[-1] if bars.get('Bars') else {}, indent=2))


Retrieved 10 bars for ES

First bar:
{
  "High": "66.98",
  "Low": "66.92",
  "Open": "66.92",
  "Close": "66.97",
  "TimeStamp": "2026-01-05T20:15:00Z",
  "TotalVolume": "5504",
  "DownTicks": 19,
  "DownVolume": 2605,
  "OpenInterest": "0",
  "IsRealtime": false,
  "IsEndOfHistory": false,
  "TotalTicks": 47,
  "UnchangedTicks": 0,
  "UnchangedVolume": 0,
  "UpTicks": 28,
  "UpVolume": 2899,
  "Epoch": 1767644100000,
  "BarStatus": "Closed"
}

Last bar:
{
  "High": "67.2",
  "Low": "67.02",
  "Open": "67.13",
  "Close": "67.06",
  "TimeStamp": "2026-01-05T21:00:00Z",
  "TotalVolume": "137216",
  "DownTicks": 403,
  "DownVolume": 66338,
  "OpenInterest": "0",
  "IsRealtime": false,
  "IsEndOfHistory": true,
  "TotalTicks": 806,
  "UnchangedTicks": 0,
  "UnchangedVolume": 0,
  "UpTicks": 403,
  "UpVolume": 70878,
  "Epoch": 1767646800000,
  "BarStatus": "Closed"
}


In [9]:
# ============================================================
# Get Symbol Details
# ============================================================

# Get detailed information about a symbol
symbol = 'ES'
details = api.market_data.get_symbol_details(symbol)

print(f"Symbol Details for {symbol}:")
print(json.dumps(details, indent=2))


Symbol Details for ES:
{
  "Symbols": [
    {
      "AssetType": "STOCK",
      "Country": "United States",
      "Currency": "USD",
      "Description": "Eversource Energy",
      "Exchange": "NYSE",
      "Symbol": "ES",
      "Root": "ES",
      "PriceFormat": {
        "Format": "Decimal",
        "Decimals": "2",
        "IncrementStyle": "Simple",
        "Increment": "0.01",
        "PointValue": "1"
      },
      "QuantityFormat": {
        "Format": "Decimal",
        "Decimals": "0",
        "IncrementStyle": "Simple",
        "Increment": "1",
        "MinimumTradeQuantity": "1"
      }
    }
  ],
  "Errors": []
}


In [ ]:
# ============================================================
# Get Real-time Quote
# ============================================================

# Get current bid/ask/last price for a symbol
# Why use this instead of bars? This is faster when you just need current price
# Bars give you historical OHLC data, quotes give you current market state

symbol = 'AAPL'
quote = api.market_data.get_quote(symbol)

print(f"Real-time Quote for {symbol}:")
print(json.dumps(quote, indent=2))

# ============================================================
# Extract and Display Key Price Information
# ============================================================

# Helper function to safely convert string/numeric values to float
# Why? The API may return prices as strings or numbers
def to_float(value, default=0.0):
    try:
        return float(value) if value is not None else default
    except (ValueError, TypeError):
        return default

# Check if we got quotes in the response
# Why check 'Quotes'? The response has a 'Quotes' array containing quote data
# If this array exists and has items, we successfully got quote data
if 'Quotes' in quote and len(quote['Quotes']) > 0:
    # Get the first quote from the array
    # Usually only one quote unless you requested multiple symbols
    q = quote['Quotes'][0]
    
    # Extract price information
    # 'Last' = most recent trade price
    last = to_float(q.get('Last', 0))
    
    # 'Bid' = highest price someone is willing to pay
    bid = to_float(q.get('Bid', 0))
    
    # 'Ask' = lowest price someone is willing to sell at
    ask = to_float(q.get('Ask', 0))
    
    # 'Volume' = total shares traded today
    volume = to_float(q.get('Volume', 0))
    
    print(f"\nKey Prices:")
    print(f"  Last: ${last:.2f}")
    print(f"  Bid:  ${bid:.2f}")
    print(f"  Ask:  ${ask:.2f}")
    print(f"  Volume: {volume:,.0f}")
    
    # Calculate and display the bid-ask spread
    # Spread = difference between ask and bid
    # Smaller spread = more liquid (easier to trade)
    # Larger spread = less liquid (may have slippage)
    print(f"  Spread: ${(ask - bid):.2f}")
else:
    # If 'Quotes' is missing or empty, something went wrong
    # Could be: invalid symbol, market closed, API error
    print("\n⚠️ No quote data returned")
    print("Check if the symbol is valid and market is open")


Real-time Quote for AAPL:
{
  "Quotes": [
    {
      "Symbol": "AAPL",
      "Open": "270.64001",
      "High": "271.51001",
      "Low": "266.14001",
      "PreviousClose": "271.01001",
      "Last": "267.27",
      "Ask": "267.27",
      "AskSize": "100",
      "Bid": "267.26",
      "BidSize": "100",
      "NetChange": "-3.74001",
      "NetChangePct": "-1.38002651636373",
      "High52Week": "288.62",
      "High52WeekTimestamp": "2025-12-03T00:00:00Z",
      "Low52Week": "169.2101",
      "Low52WeekTimestamp": "2025-04-08T00:00:00Z",
      "Volume": "45510342",
      "PreviousVolume": "37838054",
      "Close": "267.26001",
      "DailyOpenInterest": "0",
      "TradeTime": "2026-01-05T21:58:37Z",
      "TickSizeTier": "0",
      "MarketFlags": {
        "IsDelayed": false,
        "IsHardToBorrow": false,
        "IsBats": false,
        "IsHalted": false
      },
      "LastSize": "100",
      "LastVenue": "TRF",
      "VWAP": "267.880584525941"
    }
  ]
}

Key Prices:
  Last:

---
## Account Information

Retrieve account balances and positions.


In [ ]:
# ============================================================
# Get Account Balances
# ============================================================

# Get real-time account balances from TradeStation
# This shows buying power, cash balance, equity, etc.
balances = api.account.get_balances(account_id)

print(f"Account Balances for {account_id}:")
print(json.dumps(balances, indent=2))

# ============================================================
# Extract and Display Key Metrics
# ============================================================

# Note: TradeStation API may return numeric values as strings, so we convert them
# Helper function to safely convert string/numeric values to float
def to_float(value, default=0.0):
    try:
        return float(value) if value is not None else default
    except (ValueError, TypeError):
        # If conversion fails (e.g., value is None or invalid), use default
        return default

# Check if we got balance data in the response
# Why check 'Balances'? The response has a 'Balances' array with account data
# If this array exists and has items, we got valid balance information
if 'Balances' in balances and len(balances['Balances']) > 0:
    # Get the first balance record
    # Usually only one unless you have multiple account types
    balance = balances['Balances'][0]
    print(f"\nKey Metrics:")
    
    # Extract account value
    # 'AccountValue' may not always be present - it could be in 'Equity' instead
    # AccountValue = total value of cash + positions
    account_value = to_float(balance.get('AccountValue', balance.get('Equity', 0)))
    
    # Extract cash balance
    # 'CashBalance' = settled cash available (not including unsettled trades)
    cash_balance = to_float(balance.get('CashBalance', 0))
    
    # Extract buying power
    # 'BuyingPower' = maximum $ you can use to buy securities right now
    # This includes cash + margin (typically 2x or 4x cash for margin accounts)
    buying_power = to_float(balance.get('BuyingPower', 0))
    
    # Display the metrics in a readable format
    print(f"  Account Value: ${account_value:,.2f}")
    print(f"  Cash Balance: ${cash_balance:,.2f}")
    print(f"  Buying Power: ${buying_power:,.2f}")
else:
    # If 'Balances' is missing or empty, something went wrong
    # Could be: invalid account ID, API error, permissions issue
    print("\n⚠️ No balance data returned")
    print("Check if the account ID is valid")


Account Balances for SIM2977785M:
{
  "Balances": [
    {
      "AccountID": "SIM2977785M",
      "AccountType": "Margin",
      "CashBalance": "1000000",
      "BuyingPower": "4000000",
      "Equity": "1000000",
      "MarketValue": "0",
      "TodaysProfitLoss": "0",
      "UnclearedDeposit": "0",
      "BalanceDetail": {
        "CostOfPositions": "0",
        "DayTrades": "0",
        "MaintenanceRate": "0",
        "OptionBuyingPower": "1000000",
        "OptionsMarketValue": "0",
        "OvernightBuyingPower": "2000000",
        "RequiredMargin": "0",
        "UnsettledFunds": "0",
        "DayTradeExcess": "1000000",
        "RealizedProfitLoss": "0",
        "UnrealizedProfitLoss": "0"
      },
      "Commission": "0"
    }
  ],
  "Errors": []
}

Key Metrics:
  Account Value: $0.00
  Cash Balance: $1,000,000.00
  Buying Power: $4,000,000.00


In [15]:
# ============================================================
# Get Current Positions
# ============================================================

# Get all current positions
positions = api.account.get_positions(account_id)

print(f"Current Positions for {account_id}:")

# Helper function to safely convert to float for formatting
def to_float(value, default=0.0):
    try:
        return float(value) if value is not None else default
    except (ValueError, TypeError):
        return default

if 'Positions' in positions and len(positions['Positions']) > 0:
    print(f"\nFound {len(positions['Positions'])} position(s):\n")
    
    total_unrealized_pnl = 0
    
    for pos in positions['Positions']:
        symbol = pos.get('Symbol', 'N/A')
        quantity = to_float(pos.get('Quantity', 0))
        avg_price = to_float(pos.get('AveragePrice', 0))
        last_price = to_float(pos.get('Last', 0))
        unrealized_pnl = to_float(pos.get('UnrealizedProfitLoss', 0))
        
        print(f"  {symbol}:")
        print(f"    Quantity: {quantity:.0f}")
        print(f"    Avg Price: ${avg_price:.2f}")
        print(f"    Last Price: ${last_price:.2f}")
        print(f"    Unrealized P&L: ${unrealized_pnl:.2f}")
        print()
        
        total_unrealized_pnl += unrealized_pnl
    
    print(f"Total Unrealized P&L: ${total_unrealized_pnl:.2f}")
else:
    print("  No positions found.")


Current Positions for SIM2977785M:
  No positions found.


---
## Order Management

Confirm, place, and manage orders.

⚠️ **CAUTION**: Even in simulation mode, always double-check order details!


In [ ]:
# ============================================================
# Confirm Order (Dry-run - Does NOT place the order)
# ============================================================

# This validates the order and shows estimated costs/margins
# It does NOT actually place the order - safe to run

# Note: Using a stock symbol (AAPL) which works with most simulation accounts
# If your account supports futures, you can try 'ESZ24' or '@ES' instead

confirmation = api.orders.confirm_order(
    account_id=account_id,
    symbol='SNAP',         # Apple stock (use a stock your account supports)
    quantity=1,            # 1 share
    action='BUY',          # BUY or SELL
    order_type='Market',   # Market order
    time_in_force='Day'    # Day order (expires end of day)
)

print("Order Confirmation (NOT placed):")
print(json.dumps(confirmation, indent=2))

# ============================================================
# Parse and Display Key Order Details
# ============================================================

# Helper function to safely convert string/numeric values to float
# Why? TradeStation API sometimes returns numbers as strings (e.g., "123.45")
# This function handles both string and numeric types safely
def to_float(value, default=0.0):
    try:
        # Try to convert the value to float
        return float(value) if value is not None else default
    except (ValueError, TypeError):
        # If conversion fails, return the default value
        return default

# Check if confirmation was successful by looking for 'Confirmations' array
# Why check 'Confirmations'? TradeStation returns confirmations in an array
# If this key exists and has items, the order was successfully validated
if 'Confirmations' in confirmation and len(confirmation['Confirmations']) > 0:
    print(f"\n✓ Order Confirmation Successful:")
    
    # Get the first confirmation from the array
    # Usually there's only one confirmation unless it's a complex multi-leg order
    conf = confirmation['Confirmations'][0]
    
    # Display the routing option that will be used
    # 'Route' tells you which exchange/market maker will handle your order
    # 'Intelligent' is TradeStation's smart routing (recommended)
    print(f"  Route: {conf.get('Route', 'N/A')}")
    
    # Extract and format the estimated cost
    # This is the total amount of money needed to execute this trade
    # For a buy order, this is what will be debited from your account
    estimated_cost = to_float(conf.get('EstimatedCost', 0))
    print(f"  Estimated Cost: ${estimated_cost:,.2f}")
    
    # Display the order summary message from TradeStation
    # This is a human-readable description like "Buy 1 AAPL @ Market"
    # Useful for quickly verifying the order is what you intended
    summary = conf.get('SummaryMessage', 'N/A')
    print(f"  Summary: {summary}")
    
    # Check if there's an initial margin requirement
    # Why check first? Initial margin is only required for futures/options
    # For stocks, this field won't be present in the response
    # Checking before accessing prevents KeyError
    if 'InitialMarginRequirement' in conf:
        margin = to_float(conf.get('InitialMarginRequirement', 0))
        print(f"  Initial Margin: ${margin:,.2f}")
else:
    # If 'Confirmations' array is not present or is empty,
    # something went wrong with the confirmation
    # This could mean: invalid symbol, insufficient buying power, etc.
    print("\n⚠️ Order confirmation failed or returned unexpected format")
    print("Check the full JSON output above for error details")


Order Confirmation (NOT placed):
{
  "Confirmations": [
    {
      "OrderAssetCategory": "EQUITY",
      "Currency": "USD",
      "Route": "Intelligent",
      "TimeInForce": {
        "Duration": "DAY"
      },
      "AccountID": "SIM2977785M",
      "OrderConfirmID": "O6BKlhXQQkCQ7fi71/VFeA",
      "EstimatedPrice": "8.36",
      "EstimatedCost": "8.36",
      "DebitCreditEstimatedCost": "8.36",
      "EstimatedCommission": "1",
      "SummaryMessage": "Buy 1 SNAP @ Market"
    }
  ]
}


In [ ]:
# ============================================================
# Place Order (ACTUALLY PLACES THE ORDER - BE CAREFUL!)
# ============================================================

# ⚠️ WARNING: This WILL place a REAL order (or simulation order)
# Uncomment only when you're ready to actually place a trade!

# Example: Place a market order to buy AAPL stock
"""
order_response = api.orders.place_order(
    account_id=account_id,
    symbol='AAPL',         # Symbol to trade
    quantity=1,            # Number of shares
    action='BUY',          # BUY, SELL, BUYTOCOVER, SELLSHORT
    order_type='Market',   # Market, Limit, StopMarket, StopLimit
    time_in_force='DAY'    # DAY, GTC, GTD, IOC, FOK
)

print("Order Placed:")
print(json.dumps(order_response, indent=2))

# Extract the order ID for tracking/canceling
if 'Orders' in order_response and len(order_response['Orders']) > 0:
    placed_order = order_response['Orders'][0]
    order_id = placed_order.get('OrderID')
    status = placed_order.get('Status')
    
    print(f"\n✓ Order ID: {order_id}")
    print(f"✓ Status: {status}")
    print(f"✓ Symbol: {placed_order.get('Symbol')}")
    print(f"✓ Quantity: {placed_order.get('Quantity')}")
    
    # Save order_id for potential cancellation
    # last_order_id = order_id
"""

print("⚠️ PLACE ORDER CODE IS COMMENTED OUT FOR SAFETY")
print("This is intentional - uncomment only when ready to trade!")
print("")
print("To place an order:")
print("  1. Verify you're in the correct environment (sim/prod)")
print("  2. Confirm the order details with confirm_order() first")
print("  3. Uncomment the code above")
print("  4. Run the cell")


In [ ]:
# ============================================================
# Cancel Order
# ============================================================

# Cancel a pending order by its Order ID
# Note: Only works for orders that haven't been filled yet

# Example order ID (replace with actual order ID from placed order)
order_id_to_cancel = "your-order-id-here"

# Uncomment to cancel an order:
"""
cancel_response = api.orders.cancel_order(order_id_to_cancel)

print("Order Cancellation Response:")
print(json.dumps(cancel_response, indent=2))

# Check cancellation status
if 'Orders' in cancel_response and len(cancel_response['Orders']) > 0:
    canceled_order = cancel_response['Orders'][0]
    print(f"\n✓ Order {canceled_order.get('OrderID')} cancellation requested")
    print(f"✓ Status: {canceled_order.get('Status')}")
"""

print("⚠️ CANCEL ORDER CODE IS COMMENTED OUT FOR SAFETY")
print("")
print("To cancel an order:")
print("  1. Get the Order ID from a placed order")
print("  2. Set 'order_id_to_cancel' to that Order ID")
print("  3. Uncomment the code above")
print("  4. Run the cell")
print("")
print("Note: You can only cancel orders that are:")
print("  - Still pending (not filled)")
print("  - Not already canceled")
print("  - Not rejected")


In [ ]:
# ============================================================
# Get Current Orders
# ============================================================

# Retrieve all current (active/open) orders for the account
# This includes: pending, working, partially filled orders
# Does NOT include: filled, canceled, or rejected orders (use get_historical_orders for those)
orders = api.orders.get_orders(account_id)

print(f"Current Orders for {account_id}:")

# Check if we got orders in the response
# Why check 'Orders'? The response has an 'Orders' array containing order data
# If this array exists and has items, there are active orders
if 'Orders' in orders and len(orders['Orders']) > 0:
    print(f"\nFound {len(orders['Orders'])} order(s):\n")
    
    # Loop through each order and display details
    for order in orders['Orders']:
        # Extract order ID - unique identifier for this order
        order_id = order.get('OrderID', 'N/A')
        
        # Extract symbol from the order's legs
        # Why 'Legs'? Orders can have multiple legs (for spreads, etc.)
        # For simple orders, there's just one leg
        # We use [{}] as default to avoid errors if Legs is missing
        symbol = order.get('Legs', [{}])[0].get('Symbol', 'N/A')
        
        # Extract the action (BUY, SELL, etc.)
        action = order.get('Legs', [{}])[0].get('BuyOrSell', 'N/A')
        
        # Extract the quantity ordered
        quantity = order.get('Legs', [{}])[0].get('QuantityOrdered', 0)
        
        # Extract the order status
        # Common statuses: 'Received', 'Sent', 'Working', 'PartiallyFilled'
        status = order.get('Status', 'N/A')
        
        # Display order information in a structured format
        print(f"  Order ID: {order_id}")
        print(f"    Symbol: {symbol}")
        print(f"    Action: {action}")
        print(f"    Quantity: {quantity}")
        print(f"    Status: {status}")
        print()
else:
    # If 'Orders' is missing or empty, there are no active orders
    # This is normal if you haven't placed any orders yet
    print("  No active orders found.")


Current Orders for SIM2977785M:
  No active orders found.


In [ ]:
# ============================================================
# Get Order by ID
# ============================================================

# Get detailed information about a specific order
# Useful for tracking order status and fill information

# Example order ID (replace with actual order ID)
example_order_id = "your-order-id-here"

# Uncomment to get order details:
"""
order_details = api.orders.get_order(example_order_id)

print(f"Order Details for {example_order_id}:")
print(json.dumps(order_details, indent=2))

# Extract key information
if 'Orders' in order_details and len(order_details['Orders']) > 0:
    order = order_details['Orders'][0]
    
    print(f"\nOrder Summary:")
    print(f"  Order ID: {order.get('OrderID')}")
    print(f"  Status: {order.get('Status')}")
    print(f"  Symbol: {order.get('Legs', [{}])[0].get('Symbol', 'N/A')}")
    print(f"  Action: {order.get('Legs', [{}])[0].get('BuyOrSell', 'N/A')}")
    print(f"  Quantity Ordered: {order.get('Legs', [{}])[0].get('QuantityOrdered', 0)}")
    print(f"  Quantity Filled: {order.get('FilledQuantity', 0)}")
    print(f"  Fill Price: ${order.get('FilledPrice', 0)}")
"""

print("Get order by ID example - uncomment to use")
print("")
print("This is useful for:")
print("  - Checking if an order was filled")
print("  - Getting the fill price")
print("  - Monitoring order status")
print("  - Tracking partial fills")


In [ ]:
# ============================================================
# Get Historical Orders
# ============================================================

# Retrieve past orders (filled, canceled, rejected)
# This shows your complete order history for the account
# Useful for: reviewing past trades, calculating P&L, auditing

# Request historical orders with optional date filter
# 'since' parameter filters orders after this date (YYYY-MM-DD format)
# If you don't specify 'since', you'll get recent orders (usually last 30 days)
historical_orders = api.orders.get_historical_orders(
    account_id=account_id,
    since='2024-01-01'  # Optional: get orders since Jan 1, 2024
)

print(f"Historical Orders for {account_id}:")

# Check if we got historical orders in the response
# Why check 'Orders'? The response has an 'Orders' array containing past orders
# If this array exists and has items, we have order history
if 'Orders' in historical_orders and len(historical_orders['Orders']) > 0:
    print(f"\nFound {len(historical_orders['Orders'])} historical order(s):\n")
    
    # Loop through orders (show first 10 to avoid overwhelming output)
    for order in historical_orders['Orders'][:10]:
        # Extract order ID
        order_id = order.get('OrderID', 'N/A')
        
        # Extract symbol from first leg
        # Why [{}]? Provides empty dict default if Legs is missing
        symbol = order.get('Legs', [{}])[0].get('Symbol', 'N/A')
        
        # Extract action (BUY, SELL, etc.)
        action = order.get('Legs', [{}])[0].get('BuyOrSell', 'N/A')
        
        # Extract quantity
        quantity = order.get('Legs', [{}])[0].get('QuantityOrdered', 0)
        
        # Extract order status
        # Historical orders will be: 'Filled', 'Canceled', 'Rejected', etc.
        status = order.get('Status', 'N/A')
        
        # Extract date (just the date part, not the time)
        # Why [:10]? Trims the timestamp to just YYYY-MM-DD
        # Why check first? Some orders might not have OpenedDateTime
        opened = order.get('OpenedDateTime', 'N/A')[:10] if order.get('OpenedDateTime') else 'N/A'
        
        # Display order information
        print(f"  Order {order_id}:")
        print(f"    Date: {opened}")
        print(f"    Symbol: {symbol}")
        print(f"    Action: {action} {quantity}")
        print(f"    Status: {status}")
        print()
    
    # If there are more orders than we displayed, show a count
    # Why? Prevents overwhelming output while showing there's more data
    if len(historical_orders['Orders']) > 10:
        print(f"  ... and {len(historical_orders['Orders']) - 10} more orders")
else:
    # If 'Orders' is missing or empty, there's no history
    # Could mean: no orders placed yet, date filter too restrictive, or API error
    print("  No historical orders found.")

# Show usage tip for date filtering
print("\nUse 'since' parameter to filter by date:")
print("  api.orders.get_historical_orders(account_id, since='2024-01-01')")


In [ ]:
# ============================================================
# Get Available Routes
# ============================================================

# Get list of available order routing options
# What is routing? It determines which exchange/market maker receives your order
# Different routes can have different: speeds, fees, liquidity

# Request available routes from TradeStation
routes = api.orders.get_routes()

# Display full response
print("Available Order Routing Options:")
print(json.dumps(routes, indent=2))

# ============================================================
# Parse and Display Route Information
# ============================================================

# Check if we got routes in the response
# Why check 'Routes'? The response has a 'Routes' array with routing options
# If this array exists and has items, we have available routes
if 'Routes' in routes and len(routes['Routes']) > 0:
    print(f"\nFound {len(routes['Routes'])} available route(s):\n")
    
    # Loop through each routing option
    for route in routes['Routes']:
        # Extract route name (e.g., 'Intelligent', 'ARCA', 'NASDAQ')
        # This is what you pass to the 'route' parameter when placing orders
        name = route.get('Route', 'N/A')
        
        # Extract route description
        # Provides details about what this route does
        # Some routes may not have descriptions
        description = route.get('Description', 'No description available')
        
        # Display route information
        print(f"  • {name}")
        if description and description != 'No description available':
            print(f"    {description}")
        print()
else:
    # If 'Routes' is missing or empty, something went wrong
    # Could be: API error, permissions issue
    print("  No routes found")

# Display recommendation
# Why recommend 'Intelligent'? It's TradeStation's smart routing
# that automatically finds the best execution for your order
print("\nRecommended: Use 'Intelligent' routing")
print("  - TradeStation's smart order routing")
print("  - Automatically finds best execution")
print("  - Works for most strategies")
print("  - No need to manually select exchanges")


---
## Streaming Data (Optional)

**Note**: Streaming endpoints keep the connection open and continuously push data.
They work well in scripts but can be tricky in notebooks.

Example usage pattern:


In [ ]:
# ============================================================
# Stream Real-time Quotes (Commented out - use in a script)
# ============================================================

# Stream real-time bid/ask/last prices for symbols
# This provides continuous price updates as the market moves

# Uncomment to stream quotes (runs indefinitely):
"""
# Stream quotes for AAPL stock
for quote_update in api.market_data.stream_quotes('AAPL'):
    # Parse the quote data (comes as bytes)
    quote_data = json.loads(quote_update)
    
    # Extract key price info
    symbol = quote_data.get('Symbol', 'N/A')
    last = quote_data.get('Last', 'N/A')
    bid = quote_data.get('Bid', 'N/A')
    ask = quote_data.get('Ask', 'N/A')
    volume = quote_data.get('Volume', 'N/A')
    
    print(f"{symbol}: Last=${last} Bid=${bid} Ask=${ask} Vol={volume}")
    
    # Press Ctrl+C to stop streaming
    # Or add your own stop condition:
    # if some_condition:
    #     break

# Stream multiple symbols (comma-separated)
for quote_update in api.market_data.stream_quotes('AAPL,MSFT,GOOGL'):
    quote_data = json.loads(quote_update)
    print(f"{quote_data.get('Symbol')}: ${quote_data.get('Last')}")
"""

print("Real-time quote streaming is commented out (runs indefinitely)")
print("")
print("Use cases for streaming quotes:")
print("  - Live price monitoring")
print("  - Real-time bid/ask spread tracking")
print("  - Volume-based triggers")
print("  - Multi-symbol monitoring")
print("")
print("For production use, run in a separate Python script, not in a notebook")


In [ ]:
# ============================================================
# Stream Tick Bars (Commented out - use in a script)
# ============================================================

# Tick bars aggregate trades based on tick COUNT rather than TIME
# Each bar represents a fixed number of trades (e.g., 100 trades per bar)

# Uncomment to stream tick bars:
"""
# Stream tick bars for AAPL
# Parameters: symbol, interval (ticks per bar), bars_back (initial history)
for tick_bar in api.market_data.stream_tick_bars(
    symbol='AAPL',
    interval=100,      # 100 trades per bar
    bars_back=5        # Start with last 5 bars
):
    # Parse the tick bar data (comes as bytes)
    bar_data = json.loads(tick_bar)
    
    # Extract OHLC data
    timestamp = bar_data.get('TimeStamp', 'N/A')
    open_price = bar_data.get('Open', 'N/A')
    high = bar_data.get('High', 'N/A')
    low = bar_data.get('Low', 'N/A')
    close = bar_data.get('Close', 'N/A')
    volume = bar_data.get('TotalVolume', 'N/A')
    
    print(f"{timestamp}: O={open_price} H={high} L={low} C={close} V={volume}")
    
    # Press Ctrl+C to stop
"""

print("Tick bar streaming is commented out (runs indefinitely)")
print("")
print("Tick Bars vs Time Bars:")
print("  Time Bars: Fixed time interval (e.g., 5 minutes)")
print("  Tick Bars: Fixed trade count (e.g., 100 trades)")
print("")
print("Advantages of Tick Bars:")
print("  - More uniform volatility across bars")
print("  - Better for volume-based strategies")
print("  - Adjusts to market activity automatically")
print("  - Fast markets = more bars, slow markets = fewer bars")
print("")
print("Example: 100-tick bars")
print("  - Each bar = exactly 100 trades")
print("  - Time per bar varies based on trading activity")
print("")
print("For production use, run in a separate Python script")


In [20]:
# ============================================================
# Streaming Example (Commented out - use in a script instead)
# ============================================================

# Streaming is better suited for standalone scripts
# Uncomment and run in a Python script for live streaming:

# Stream real-time bar data
for bar_data in api.market_data.stream_bars(
    symbol='@ES',
    interval=1,
    unit='Minute'
):
    print(f"New bar: {bar_data}")
    # Process each bar as it arrives
    # Press Ctrl+C to stop

# Stream real-time positions
for position_update in api.account.stream_positions(account_id):
    print(f"Position update: {position_update}")
    # Process position updates

# Stream real-time order updates
for order_update in api.orders.stream_orders(account_id):
    print(f"Order update: {order_update}")
    # Process order status changes

print("Streaming examples are commented out.")
print("See the code cell above for usage patterns.")
print("Run streaming code in a Python script for best results.")


New bar: b'{"High":"6945.5","Low":"6945","Open":"6945","Close":"6945.25","TimeStamp":"2026-01-05T21:36:00Z","TotalVolume":"65","DownTicks":5,"DownVolume":8,"OpenInterest":"0","IsRealtime":false,"IsEndOfHistory":false,"TotalTicks":34,"UnchangedTicks":0,"UnchangedVolume":0,"UpTicks":29,"UpVolume":57,"Epoch":1767648960000,"BarStatus":"Closed"}'
New bar: b'{"High":"6945.5","Low":"6945.25","Open":"6945.5","Close":"6945.5","TimeStamp":"2026-01-05T21:37:00Z","TotalVolume":"110","DownTicks":11,"DownVolume":14,"OpenInterest":"0","IsRealtime":false,"IsEndOfHistory":false,"TotalTicks":66,"UnchangedTicks":0,"UnchangedVolume":0,"UpTicks":55,"UpVolume":96,"Epoch":1767649020000,"BarStatus":"Closed"}'
New bar: b'{"High":"6945.5","Low":"6945.25","Open":"6945.5","Close":"6945.25","TimeStamp":"2026-01-05T21:38:00Z","TotalVolume":"54","DownTicks":31,"DownVolume":49,"OpenInterest":"0","IsRealtime":false,"IsEndOfHistory":false,"TotalTicks":36,"UnchangedTicks":0,"UnchangedVolume":0,"UpTicks":5,"UpVolume":5,"

KeyboardInterrupt: 

---
## Additional Resources

- **Library Documentation**: See `api/README.md` and `api/USAGE.md`
- **Code Examples**: See `api/example.py` for standalone Python examples
- **API Specification**: See `api_spec/openapi.json` for TradeStation API details
- **Original Demo**: See `ts_api_demo.ipynb` for raw API call examples

---

**Happy Trading! 📈**
